# Práctica: Inferencia Estadística

En este cuaderno construimos intervalos de confianza, realizamos contrastes de hipótesis sobre medias y proporciones, y verificamos por simulación qué significa realmente un nivel de confianza del 95%.

In [ ]:
import sys
sys.path.append("../../src")

import numpy as np
import matplotlib.pyplot as plt
from stats_toolkit import inference as inf

## 1. Intervalo de confianza para una media

Simulamos una muestra de 40 tiempos de respuesta de una API (población Normal con μ=120ms, σ=15ms) y construimos su IC al 95%, sabiendo σ (caso z) y sin saberlo (caso t, estimando s a partir de la propia muestra).

In [ ]:
rng = np.random.default_rng(1)
muestra = rng.normal(loc=120, scale=15, size=40)

x_bar = muestra.mean()
s = muestra.std(ddof=1)
print(f"Media muestral: {x_bar:.2f} ms | Desviación muestral: {s:.2f} ms")

ic_z = inf.confidence_interval_mean_z(x_bar, sigma=15, n=40, confidence=0.95)
ic_t = inf.confidence_interval_mean_t(x_bar, sample_std=s, n=40, confidence=0.95)

print(f"IC 95% (sigma conocida, z): ({ic_z[0]:.2f}, {ic_z[1]:.2f})")
print(f"IC 95% (sigma desconocida, t): ({ic_t[0]:.2f}, {ic_t[1]:.2f})")

Observa que el intervalo basado en la t es ligeramente más ancho: refleja la incertidumbre extra de tener que estimar σ desde la propia muestra.

## 2. Intervalo de confianza para una proporción

Retomamos el ejemplo de conversión del Módulo 3: 15 conversiones sobre 50 clientes.

In [ ]:
p_hat = 15 / 50
lower, upper = inf.confidence_interval_proportion(p_hat, n=50, confidence=0.95)
print(f"Tasa de conversión observada: {p_hat:.3f}")
print(f"IC 95%: ({lower:.3f}, {upper:.3f})")

## 3. Contraste de hipótesis: test t de una muestra

¿El salario medio de un departamento de 10 empleados es distinto de los 2400€ que afirma la dirección?

In [ ]:
salarios = [2340, 2510, 2280, 2600, 2455, 2390, 2470, 2530, 2410, 2380]

t_stat, p_value = inf.t_test_mean(salarios, mu0=2400)
print(f"t = {t_stat:.3f}, p-valor = {p_value:.4f}")

alpha = 0.05
if p_value < alpha:
    print(f"p-valor < {alpha}: rechazamos H0, hay evidencia de que el salario medio difiere de 2400€")
else:
    print(f"p-valor >= {alpha}: no hay evidencia suficiente para rechazar H0")

## 4. Contraste de hipótesis: A/B test de dos proporciones

Comparamos dos variantes de una landing page: variante A (15/50 conversiones) vs. variante B (27/60 conversiones).

In [ ]:
z_stat, p_value = inf.two_proportion_z_test(x1=15, n1=50, x2=27, n2=60)
print(f"z = {z_stat:.3f}, p-valor = {p_value:.4f}")

p_a, p_b = 15/50, 27/60
print(f"Tasa A: {p_a:.3f} | Tasa B: {p_b:.3f}")

if p_value < 0.05:
    print("La diferencia entre variantes es estadísticamente significativa (p < 0.05).")
else:
    print("No hay evidencia suficiente de diferencia real entre variantes.")

## 5. ¿Qué significa un intervalo de confianza del 95%?

Simulamos 5000 muestras distintas de una población Normal(μ=100, σ=15), construimos un IC al 95% para cada una, y visualizamos cuántos contienen realmente μ=100. Dibujamos los primeros 100 para verlo de forma intuitiva.

In [ ]:
true_mu = 100
n_intervals = 100

rng = np.random.default_rng(7)
fig, ax = plt.subplots(figsize=(9, 8))

contains_mu = 0
for i in range(n_intervals):
    sample = rng.normal(true_mu, 15, size=30)
    x_bar = sample.mean()
    lower, upper = inf.confidence_interval_mean_z(x_bar, sigma=15, n=30, confidence=0.95)
    color = "steelblue" if lower <= true_mu <= upper else "crimson"
    if lower <= true_mu <= upper:
        contains_mu += 1
    ax.plot([lower, upper], [i, i], color=color, alpha=0.7)

ax.axvline(true_mu, color="black", linestyle="--", label=f"Verdadera μ = {true_mu}")
ax.set_yticks([])
ax.set_xlabel("Valor")
ax.set_title(f"100 intervalos de confianza al 95% — {contains_mu} de 100 contienen μ (línea roja = no la contiene)")
ax.legend()
plt.show()

# Ahora con 5000 simulaciones, usando la función del toolkit
coverage = inf.simulate_ci_coverage(true_mu=100, sigma=15, n=30, confidence=0.95, n_simulations=5000, seed=42)
print(f"Cobertura empírica sobre 5000 simulaciones: {coverage:.3f} (esperado ≈ 0.95)")

## Ejercicios propuestos

1. Repite la simulación de la sección 5 con un nivel de confianza del 90% y del 99%. ¿Cómo cambia el ancho de los intervalos y la proporción que contiene a μ?
2. Una muestra de 12 tiempos de entrega (en días) tiene media 5.4 y desviación muestral 1.2. Construye un IC al 95% para la media poblacional usando `confidence_interval_mean_t`, y contrasta si la media real podría ser de 6 días con `t_test_mean` (necesitarás generar o simular una muestra con esos estadísticos, o adaptar la función a partir de `sample_mean`/`sample_std` directamente).
3. En el A/B test de la sección 4, calcula también el intervalo de confianza para la diferencia de proporciones ($\hat{p}_B - \hat{p}_A$) en vez de solo el contraste de hipótesis. ¿Qué información adicional aporta el intervalo frente al p-valor?